# Pré-processamento do mosaico Sentinel-2 (estágio 05)

Constrói o composite multiespectral alinhado e normalizado a partir do mosaico livre de nuvens exportado no estágio 01, alinhando-o ao grid da máscara binária final do estágio 04 (sobreposição pixel a pixel) e normalizando a reflectância para o intervalo [0, 1]. O composite é persistido em `MyDrive/tcc/data/processed/composites/` e é a entrada do estágio 06 (geração de patches). O mosaico e a máscara final são reutilizados dos estágios 01 e 04 (idempotência); a figura de registro vai para `MyDrive/tcc/artifacts/figures/`.

## Bootstrap do workspace

O primeiro passo baixa e executa `src/bootstrap.py` (somente stdlib) — necessário porque o `src/` ainda não está disponível para import em uma sessão nova. O bootstrap obtém o repositório público, extrai `src/`, `data/external/` e `requirements-runtime.txt` para o workspace e adiciona o workspace ao `sys.path`. O `reload` garante que reexecuções usem a versão mais recente baixada.

In [ ]:
# Baixa e executa o bootstrap do workspace (etapa prévia ao import de src/).
import importlib
import pathlib
import sys
import urllib.request

BOOTSTRAP_URL = "https://raw.githubusercontent.com/oguel/tcc-umamba/main/src/bootstrap.py"
pathlib.Path("bootstrap.py").write_bytes(urllib.request.urlopen(BOOTSTRAP_URL).read())
sys.path.insert(0, str(pathlib.Path.cwd()))

# Recarrega o módulo para não reutilizar uma versão antiga em cache no kernel.
bootstrap = importlib.import_module("bootstrap")
importlib.reload(bootstrap)

workspace = bootstrap.bootstrap_workspace()
print(f"Workspace: {workspace}")

## Dependências pinadas

Instala as versões fixadas em `requirements-runtime.txt` (incluindo rasterio, usado na leitura e na escrita dos GeoTIFFs), garantindo o mesmo conjunto de bibliotecas nas duas plataformas.

In [ ]:
# Instala as versões pinadas do requirements-runtime.txt no ambiente atual.
import subprocess
import sys

requirements = pathlib.Path(workspace) / "requirements-runtime.txt"
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-r", str(requirements)],
    check=True,
)
print("Dependências instaladas a partir de:", requirements)

## Pacote compartilhado, plataforma e armazenamento

Importa o pacote `src/` (já entregue pelo bootstrap) e identifica a plataforma pela abstração em `src/io.py`. Em seguida garante a raiz `MyDrive/tcc/` e resolve todos os caminhos de armazenamento definidos em `src/config.yaml`.

In [ ]:
# Importa o pacote compartilhado, identifica a plataforma e resolve os caminhos de armazenamento.
from src import io
from src.config import get_config

platform = io.detect_platform()
storage_paths = io.resolve_storage_paths()
config = get_config()
print(f"Plataforma: {platform}")
print(f"Composite (alinhado e normalizado): {storage_paths['data_processed_composites']}")
print(f"Figuras: {storage_paths['artifacts_figures']}")

## Reprodutibilidade

Fixa as sementes de python/numpy/torch/cuda e habilita as flags determinísticas do PyTorch, garantindo o mesmo protocolo de execução nas duas plataformas.

In [ ]:
# Fixa sementes e flags determinísticas do PyTorch de acordo com a configuração.
from src.utils import set_all_seeds, set_deterministic_flags

set_all_seeds(config["reproducibility"]["seed"])
set_deterministic_flags()
print(f"Seed fixada: {config['reproducibility']['seed']}")

## Dependências dos estágios 01 e 04

Verifica que o mosaico Sentinel-2 exportado no estágio 01 e a máscara binária final do estágio 04 estão disponíveis no caminho canônico — sem estas dependências o pré-processamento não pode prosseguir.

In [ ]:
# Verifica as dependências do estágio 01 (mosaico) e do estágio 04 (máscara final).
from src.data.mask_finalization import final_mask_path
from src.data.preprocessing import mosaic_path

dependencies = {
    "Mosaico Sentinel-2 (estágio 01)": mosaic_path(storage_paths),
    "Máscara binária final (estágio 04)": final_mask_path(storage_paths),
}
for name, path in dependencies.items():
    if not io.path_exists(path):
        raise FileNotFoundError(f"Dependência não encontrada: {path}")
    print(f"Disponível: {name}: {path}")

## Composite alinhado e normalizado

Garante (idempotente) o composite multiespectral em `MyDrive/tcc/data/processed/composites/`: o mosaico do estágio 01 é alinhado ao grid da máscara final do estágio 04 (reamostrado bilinear quando necessário) e a reflectância é normalizada para [0, 1] pela escala do Sentinel-2. Execuções repetidas reutilizam o composite já existente.

In [ ]:
# Garante o composite alinhado e normalizado (reutiliza se já existir).
from src.data.preprocessing import build_preprocessed_composite

composite_file = build_preprocessed_composite(storage_paths)

## Verificação do composite

Confere o grid, o CRS, o número de bandas, o alinhamento à máscara final e o intervalo de reflectância do composite persistido.

In [ ]:
# Verifica o composite persistido (grid, CRS, bandas, alinhamento e intervalo).
from src.data.preprocessing import verify_composite

composite_stats = verify_composite(storage_paths)
vmin, vmax = composite_stats["value_range"]
print(f"Grid: {composite_stats['shape'][0]}x{composite_stats['shape'][1]} px ({composite_stats['crs']})")
print(f"Bandas: {composite_stats['bands']}")
print(f"Alinhado à máscara final: {composite_stats['aligned_to_mask']}")
print(f"Reflectância: [{vmin:.3f}, {vmax:.3f}]")

## Figura do composite

Renderiza e persiste a miniatura RGB do composite (bandas B4, B3, B2, com realce por percentil) em `MyDrive/tcc/artifacts/figures/`; execuções repetidas reutilizam a figura já existente (idempotência).

In [ ]:
# Renderiza e persiste a figura do composite (reutiliza se já existir).
from src.data.preprocessing import save_composite_preview

figure_path = save_composite_preview(storage_paths)

## Resumo da etapa

Exibe o resumo do pré-processamento: dependências reutilizadas, composite persistido, métricas de verificação e figura de registro.

In [ ]:
# Exibe o resumo da etapa de pré-processamento do composite.
summary = {
    "Mosaico de entrada (estágio 01)": str(dependencies["Mosaico Sentinel-2 (estágio 01)"]),
    "Máscara de referência (estágio 04)": str(dependencies["Máscara binária final (estágio 04)"]),
    "Composite": str(composite_file),
    "Bandas": composite_stats["bands"],
    "Grid": f"{composite_stats['shape'][0]}x{composite_stats['shape'][1]} px",
    "Alinhado à máscara final": composite_stats["aligned_to_mask"],
    "Figura": str(figure_path),
}
for key, value in summary.items():
    print(f"{key}: {value}")
print("Estágio 05 concluído.")